# Module 9: Frontiers — RL Trading Environment

## Learning Objectives
By completing this notebook, you will:
1. Build a stock trading environment that conforms to the Gymnasium API
2. Design a meaningful state representation (price history + position) and reward (portfolio returns)
3. Train a tabular Q-learning agent on synthetic price data
4. Backtest the agent and compare its equity curve to a buy-and-hold benchmark

## Prerequisites
- Module 3: Q-learning
- Module 4: Discretization of continuous state spaces
- Basic finance concepts: portfolio value, returns, drawdown

## Estimated Time: 15 minutes

---

## RL for Sequential Trading Decisions

Trading is a natural RL problem:
- **State**: observable market features (prices, technical indicators, current position)
- **Action**: buy / hold / sell
- **Reward**: portfolio return from this action
- **Goal**: maximize cumulative return (or risk-adjusted return)

The key challenge: financial markets are **non-stationary** and reward functions are **delayed** — a buy decision only pays off when you later sell.

We use synthetic price data (random walk with drift) to keep results reproducible and fast. The same framework generalizes to real market data.

In [ ]:
learning_objectives([
    "Module 3: Q-learning",
    "Module 4: Discretization of continuous state spaces",
    "Basic finance concepts: portfolio value, returns, drawdown",
    "## RL for Sequential Trading Decisions",
    "**State**: observable market features (prices, technical indicators, current position)",
    "**Action**: buy / hold / sell",
    "**Reward**: portfolio return from this action",
    "**Goal**: maximize cumulative return (or risk-adjusted return)",
    "and reward functions are **delayed** — a buy decision only pays off when you later sell."
])

In [ ]:
import sys; sys.path.insert(0, '../../../../..')
from resources.notebook_style import apply_course_theme, learning_objectives, section_divider, callout, key_takeaways
from resources.graphics.plot_theme import apply_plot_theme

In [ ]:
apply_course_theme()
apply_plot_theme()

In [ ]:
# Purpose: Import dependencies
# Key Concept: We implement Gymnasium API manually to avoid dependency issues

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from collections import deque
import warnings

warnings.filterwarnings('ignore')

SEED = 42
np.random.seed(SEED)

print("RL Trading: Sequential portfolio management with Q-learning")

## 1. Synthetic Price Data

We generate a random walk with drift and volatility — a simplified Geometric Brownian Motion (GBM) with discrete steps:

$$P_t = P_{t-1} \cdot \exp(\mu + \sigma \cdot \epsilon_t)$$

where $\mu$ is the daily drift (slight positive bias) and $\sigma$ is volatility. This gives log-normally distributed returns, matching real financial data qualitatively.

In [ ]:
# Purpose: Generate synthetic stock price data using a random walk with drift
# Key Concept: Geometric Brownian Motion gives positive-only prices and log-normal returns

def generate_price_series(
    n_steps: int = 1000,
    initial_price: float = 100.0,
    mu: float = 0.0005,     # daily drift (~12.5% annual)
    sigma: float = 0.015,   # daily volatility (~24% annual)
    seed: int = SEED
) -> np.ndarray:
    """
    Generate a price series using Geometric Brownian Motion.

    Parameters
    ----------
    n_steps : int
        Number of time steps.
    initial_price : float
        Starting price.
    mu : float
        Drift per step.
    sigma : float
        Volatility per step.
    seed : int
        Random seed.

    Returns
    -------
    np.ndarray
        Price series of shape (n_steps + 1,).
    """
    np.random.seed(seed)
    returns = np.random.normal(mu, sigma, n_steps)
    prices = initial_price * np.exp(np.cumsum(returns))
    return np.concatenate([[initial_price], prices])


# Generate training and testing price series
TRAIN_PRICES = generate_price_series(n_steps=1500, seed=SEED)
TEST_PRICES = generate_price_series(n_steps=500, seed=SEED + 100, mu=0.0003)

# Visualize training data
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

ax = axes[0]
ax.plot(TRAIN_PRICES, color='steelblue', linewidth=1.5)
ax.set_xlabel('Time Step', fontsize=12)
ax.set_ylabel('Price', fontsize=12)
ax.set_title(f'Training Price Series\n(drift={0.0005}, vol={0.015})', fontsize=12)
ax.grid(True, alpha=0.3)

ax = axes[1]
returns = np.diff(np.log(TRAIN_PRICES))
ax.hist(returns, bins=50, color='steelblue', edgecolor='white', alpha=0.8)
ax.set_xlabel('Log Return', fontsize=12)
ax.set_ylabel('Frequency', fontsize=12)
ax.set_title('Return Distribution (should be ~Normal)', fontsize=12)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()
print(f"Training prices: {len(TRAIN_PRICES)} steps | Test prices: {len(TEST_PRICES)} steps")

## 2. Trading Environment

The environment follows the Gymnasium API (`reset`, `step`, observation/action spaces). The state space is discretized to enable tabular Q-learning:
- **Position**: {flat (no position), long (holding stock)}
- **Recent price trend**: discretized momentum signal (past N returns → binned)

Actions: 0=stay flat / sell if long, 1=buy / stay long

In [ ]:
# Purpose: Implement the trading environment following Gymnasium API conventions
# Key Concept: State discretization converts continuous price history to tabular Q-learning compatible form

class TradingEnv:
    """
    Simple stock trading environment.

    State: (position, price_trend_bin)
      - position: 0=flat, 1=long
      - price_trend_bin: discretized recent momentum (0..n_bins-1)

    Actions: 0=close/stay_flat, 1=open/stay_long

    Reward: portfolio log-return this step
      - flat: 0 (no exposure)
      - long: log_return of the stock (positive = profit, negative = loss)

    Parameters
    ----------
    prices : np.ndarray
        Price series to trade on.
    lookback : int
        Number of past returns to use as state features.
    n_bins : int
        Number of bins for discretizing momentum.
    transaction_cost : float
        Cost (as fraction) per buy or sell (e.g., 0.001 = 0.1 bps).
    """

    # Action constants
    FLAT = 0
    LONG = 1
    N_ACTIONS = 2

    def __init__(
        self,
        prices: np.ndarray,
        lookback: int = 5,
        n_bins: int = 5,
        transaction_cost: float = 0.001
    ):
        self.prices = prices
        self.lookback = lookback
        self.n_bins = n_bins
        self.transaction_cost = transaction_cost

        # Precompute log returns
        self.log_returns = np.diff(np.log(prices))

        # Compute bin edges from return distribution
        self.bin_edges = np.percentile(self.log_returns, np.linspace(0, 100, n_bins + 1))

        # State space size: 2 positions x n_bins trend bins
        self.n_states = 2 * n_bins

        # Episode limits
        self.max_steps = len(self.log_returns) - lookback - 1

        self._t = lookback
        self._position = self.FLAT
        self._portfolio_value = 1.0

    def _get_obs(self) -> int:
        """
        Compute discrete state index from current position and recent returns.

        Returns
        -------
        int
            Flat state index in [0, n_states).
        """
        # Recent momentum: sum of last `lookback` log returns
        recent_returns = self.log_returns[self._t - self.lookback: self._t]
        momentum = recent_returns.mean()

        # Discretize momentum into n_bins bins
        trend_bin = np.digitize(momentum, self.bin_edges[1:-1])  # 0..n_bins-1
        trend_bin = min(trend_bin, self.n_bins - 1)

        return self._position * self.n_bins + trend_bin

    def reset(self) -> tuple:
        """
        Reset environment to start of price series.

        Returns
        -------
        tuple of (state_idx, info_dict)
        """
        self._t = self.lookback
        self._position = self.FLAT
        self._portfolio_value = 1.0
        return self._get_obs(), {}

    def step(self, action: int) -> tuple:
        """
        Execute one trading step.

        Parameters
        ----------
        action : int
            0 = flat (sell if currently long), 1 = long (buy if currently flat)

        Returns
        -------
        tuple of (next_state, reward, terminated, truncated, info)
        """
        old_position = self._position

        # Compute transaction cost for position change
        tc = 0.0
        if action != old_position:
            tc = self.transaction_cost  # paid on position change

        # Execute action: update position
        self._position = action

        # Observe next log return
        log_ret = self.log_returns[self._t]
        self._t += 1

        # Reward: portfolio log return (0 if flat, stock return if long)
        if self._position == self.LONG:
            reward = log_ret - tc
        else:
            reward = -tc  # Only pay cost if we transacted to flat

        # Update portfolio value
        self._portfolio_value *= np.exp(reward)

        terminated = self._t >= len(self.log_returns)
        truncated = False

        next_state = self._get_obs() if not terminated else 0
        info = {
            'portfolio_value': self._portfolio_value,
            'price': self.prices[self._t] if self._t < len(self.prices) else self.prices[-1],
            'position': self._position
        }

        return next_state, reward, terminated, truncated, info


# Sanity check
env = TradingEnv(TRAIN_PRICES)
state, _ = env.reset()
print(f"Initial state: {state} | n_states: {env.n_states} | n_actions: {env.N_ACTIONS}")
next_s, r, term, trunc, info = env.step(TradingEnv.LONG)
print(f"After buying: reward={r:.4f}, portfolio_value={info['portfolio_value']:.4f}")

## 3. Q-Learning Agent for Trading

We use tabular Q-learning with epsilon-greedy exploration. The Q-table has shape `(n_states, n_actions) = (10, 2)` — small enough to train quickly.

In [ ]:
# Purpose: Train tabular Q-learning agent on the trading environment
# Key Concept: The agent learns which momentum signal to buy on and which to stay flat

def train_trading_agent(
    prices: np.ndarray,
    n_episodes: int = 100,
    alpha: float = 0.1,
    gamma: float = 0.99,
    epsilon_start: float = 1.0,
    epsilon_end: float = 0.05,
    lookback: int = 5,
    n_bins: int = 5,
    transaction_cost: float = 0.001,
    seed: int = SEED
) -> dict:
    """
    Train a Q-learning agent on the trading environment.

    Returns
    -------
    dict with keys: 'Q', 'episode_returns', 'epsilon_history'
    """
    np.random.seed(seed)
    env = TradingEnv(prices, lookback=lookback, n_bins=n_bins,
                     transaction_cost=transaction_cost)
    Q = np.zeros((env.n_states, env.N_ACTIONS))

    episode_returns = []
    epsilon_history = []

    for ep in range(n_episodes):
        # Linear epsilon decay
        epsilon = max(epsilon_end, epsilon_start - (epsilon_start - epsilon_end) * ep / n_episodes)
        epsilon_history.append(epsilon)

        state, _ = env.reset()
        ep_return = 0.0
        done = False

        while not done:
            # Epsilon-greedy action selection
            if np.random.random() < epsilon:
                action = np.random.randint(env.N_ACTIONS)
            else:
                action = int(np.argmax(Q[state]))

            next_state, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated

            # Q-learning update
            if not done:
                td_target = reward + gamma * np.max(Q[next_state])
            else:
                td_target = reward

            Q[state, action] += alpha * (td_target - Q[state, action])
            state = next_state
            ep_return += reward

        episode_returns.append(ep_return)

    return {
        'Q': Q,
        'episode_returns': episode_returns,
        'epsilon_history': epsilon_history
    }


print("Training Q-learning trading agent...")
train_results = train_trading_agent(TRAIN_PRICES, n_episodes=200)
Q_trained = train_results['Q']

print(f"Training episodes: {len(train_results['episode_returns'])}")
print(f"Final episode log-return: {train_results['episode_returns'][-1]:.4f}")
print(f"\nLearned Q-table (rows=states, cols=actions [flat, long]):")
print(Q_trained.round(4))

## 4. Backtesting on Test Data

We evaluate the trained agent on the held-out test price series. No learning occurs during backtesting — the Q-table is fixed (greedy policy). We compare against a buy-and-hold baseline.

In [ ]:
# Purpose: Backtest the trained agent on out-of-sample test data
# Key Concept: Test performance on unseen data is the only honest measure of trading strategy quality

def backtest(
    prices: np.ndarray,
    Q: np.ndarray,
    lookback: int = 5,
    n_bins: int = 5,
    transaction_cost: float = 0.001
) -> dict:
    """
    Run greedy policy on price series and record full equity curve.

    Returns
    -------
    dict with keys: 'equity_curve', 'positions', 'prices', 'rewards'
    """
    env = TradingEnv(prices, lookback=lookback, n_bins=n_bins,
                     transaction_cost=transaction_cost)
    state, _ = env.reset()

    equity_curve = [1.0]
    positions = []
    rewards = []
    step_prices = [prices[lookback]]
    done = False

    while not done:
        action = int(np.argmax(Q[state]))
        next_state, reward, terminated, truncated, info = env.step(action)
        done = terminated or truncated

        equity_curve.append(info['portfolio_value'])
        positions.append(info['position'])
        rewards.append(reward)
        step_prices.append(info['price'])
        state = next_state

    return {
        'equity_curve': np.array(equity_curve),
        'positions': np.array(positions),
        'prices': np.array(step_prices),
        'rewards': np.array(rewards)
    }


# Backtest trained agent
bt_results = backtest(TEST_PRICES, Q_trained)

# Compute buy-and-hold baseline
# Always long from the start — normalized to start at 1.0
bah_equity = TEST_PRICES / TEST_PRICES[0]

# Summary statistics
agent_return = bt_results['equity_curve'][-1] - 1.0
bah_return = bah_equity[-1] - 1.0
long_fraction = bt_results['positions'].mean()

print("Backtest Results (out-of-sample):")
print("-" * 40)
print(f"RL Agent total return: {agent_return:+.2%}")
print(f"Buy-and-hold return:   {bah_return:+.2%}")
print(f"Agent long fraction:   {long_fraction:.1%}")

agent_rewards = bt_results['rewards']
sharpe = (agent_rewards.mean() / (agent_rewards.std() + 1e-8)) * np.sqrt(252)
print(f"Agent Sharpe (annualized): {sharpe:.2f}")

## 5. Visualizing the Equity Curve

In [ ]:
# Purpose: Visualize equity curve, price series, and position over time
# Key Concept: A good trading agent should capture gains while limiting downside exposure

fig = plt.figure(figsize=(14, 9))
gs = gridspec.GridSpec(3, 1, height_ratios=[2, 1, 1], hspace=0.35)

# --- Plot 1: Equity curves ---
ax1 = fig.add_subplot(gs[0])
t_range = range(len(bt_results['equity_curve']))
ax1.plot(t_range, bt_results['equity_curve'],
         color='darkorange', linewidth=2, label='RL Agent')
# Align buy-and-hold to the lookback-trimmed test period
bah_trimmed = bah_equity[5:5 + len(bt_results['equity_curve'])]
if len(bah_trimmed) < len(bt_results['equity_curve']):
    bah_trimmed = np.pad(bah_trimmed, (0, len(bt_results['equity_curve']) - len(bah_trimmed)),
                         'edge')
ax1.plot(t_range, bah_trimmed[:len(t_range)],
         color='steelblue', linewidth=2, linestyle='--', label='Buy-and-Hold')
ax1.axhline(y=1.0, color='black', linewidth=0.8, alpha=0.5)
ax1.set_ylabel('Portfolio Value (normalized)', fontsize=11)
ax1.set_title('Backtest: RL Agent vs Buy-and-Hold (out-of-sample)', fontsize=13)
ax1.legend(fontsize=10)
ax1.grid(True, alpha=0.3)

# --- Plot 2: Price series ---
ax2 = fig.add_subplot(gs[1])
ax2.plot(bt_results['prices'], color='gray', linewidth=1.5, alpha=0.8)
ax2.set_ylabel('Stock Price', fontsize=11)
ax2.grid(True, alpha=0.3)

# --- Plot 3: Positions ---
ax3 = fig.add_subplot(gs[2])
position_series = np.zeros(len(bt_results['equity_curve']))
position_series[1:] = bt_results['positions']
ax3.fill_between(range(len(position_series)), position_series,
                  alpha=0.7, color='darkorange', label='Long (1) / Flat (0)')
ax3.set_yticks([0, 1])
ax3.set_yticklabels(['Flat', 'Long'])
ax3.set_xlabel('Test Step', fontsize=11)
ax3.set_ylabel('Position', fontsize=11)
ax3.grid(True, alpha=0.3)

plt.savefig('rl_trading_backtest.png', dpi=100, bbox_inches='tight')
plt.show()
print("Figure saved to rl_trading_backtest.png")

## 6. Training Curve and Q-Table Analysis

In [ ]:
# Purpose: Visualize Q-table structure and training dynamics
# Key Concept: The Q-table reveals what the agent learned — which states prefer long vs. flat

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# --- Plot 1: Training episode returns ---
ax = axes[0]
ep_returns = train_results['episode_returns']
smoothed = np.convolve(ep_returns, np.ones(20) / 20, mode='valid')
ax.plot(ep_returns, color='gray', alpha=0.4, linewidth=1, label='Raw')
ax.plot(range(19, len(ep_returns)), smoothed,
        color='darkorange', linewidth=2, label='20-ep avg')
ax.set_xlabel('Training Episode', fontsize=11)
ax.set_ylabel('Episode Log Return', fontsize=11)
ax.set_title('Training Curve', fontsize=12)
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

# --- Plot 2: Q-table heatmap ---
ax = axes[1]
n_states = Q_trained.shape[0]
n_bins = n_states // 2
im = ax.imshow(Q_trained, cmap='RdYlGn', aspect='auto')
plt.colorbar(im, ax=ax, label='Q value')
ax.set_xticks([0, 1])
ax.set_xticklabels(['Flat', 'Long'], fontsize=10)
ax.set_yticks(range(n_states))
pos_labels = ['Flat'] * n_bins + ['Long'] * n_bins
bin_labels = [f'bin{i%n_bins}' for i in range(n_states)]
ax.set_yticklabels([f'{p}|{b}' for p, b in zip(pos_labels, bin_labels)], fontsize=7)
ax.set_xlabel('Action', fontsize=11)
ax.set_ylabel('State (position|trend_bin)', fontsize=11)
ax.set_title('Learned Q-Table', fontsize=12)

# --- Plot 3: Preferred action per state ---
ax = axes[2]
preferred_actions = np.argmax(Q_trained, axis=1)
preferred_labels = ['Flat' if a == 0 else 'Long' for a in preferred_actions]
colors = ['steelblue' if a == 0 else 'darkorange' for a in preferred_actions]

bars = ax.barh(range(n_states), [1] * n_states, color=colors, edgecolor='white')
ax.set_yticks(range(n_states))
ax.set_yticklabels([f'{p}|{b}' for p, b in zip(pos_labels, bin_labels)], fontsize=7)
ax.set_xticks([])
ax.set_title('Preferred Action per State', fontsize=12)

# Legend
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor='steelblue', label='Flat'),
                   Patch(facecolor='darkorange', label='Long')]
ax.legend(handles=legend_elements, fontsize=9)

plt.tight_layout()
plt.savefig('rl_trading_analysis.png', dpi=100, bbox_inches='tight')
plt.show()
print("Figure saved to rl_trading_analysis.png")

## Exercise 9.3: Transaction Cost Sensitivity

**Task:** Transaction costs penalize frequent trading. Train agents with `transaction_cost` in `{0.0, 0.001, 0.005, 0.01}` for 150 episodes each. Report:
1. Final test portfolio value for each cost level
2. Average long fraction (how often the agent is in the market)

**Expected observation:** Higher transaction costs should cause the agent to trade less frequently (lower volatility of positions) and potentially have lower returns.

<details>
<summary>Hint 1</summary>
Call `train_trading_agent(TRAIN_PRICES, transaction_cost=tc, n_episodes=150)` then `backtest(TEST_PRICES, results['Q'], transaction_cost=tc)`.
</details>

In [ ]:
# YOUR CODE HERE
# ---------------
transaction_costs = [0.0, 0.001, 0.005, 0.01]
tc_results = {}  # Map tc -> {'portfolio_value': float, 'long_fraction': float}

# Fill tc_results
# Print a summary table

In [ ]:
# AUTO-GRADED TESTS - Do not modify
# ----------------------------------
def test_exercise_9_3():
    assert len(tc_results) == 4, \
        f"Expected 4 transaction cost experiments, got {len(tc_results)}"
    for tc in transaction_costs:
        assert tc in tc_results, f"Missing result for transaction_cost={tc}"
        d = tc_results[tc]
        assert 'portfolio_value' in d, \
            f"tc_results[{tc}] must have 'portfolio_value' key"
        assert 'long_fraction' in d, \
            f"tc_results[{tc}] must have 'long_fraction' key"
        pv = d['portfolio_value']
        lf = d['long_fraction']
        assert pv > 0, f"Portfolio value must be positive, got {pv}"
        assert 0.0 <= lf <= 1.0, \
            f"Long fraction must be in [0, 1], got {lf}"
    print("Exercise 9.3 passed! Transaction cost sensitivity analysis complete.")

test_exercise_9_3()

## Exercise 9.4: Reward Shaping for Drawdown Penalty

**Task:** Portfolio drawdown measures peak-to-trough loss. Modify the reward to penalize large drawdowns:

$$r_t^{\text{shaped}} = r_t - \lambda \cdot \max(0, \text{drawdown}_t)$$

where $\text{drawdown}_t = (\text{peak}_t - V_t) / \text{peak}_t$ and $\lambda = 0.1$.

Implement `TradingEnvWithDrawdown` as a subclass of `TradingEnv` that overrides `step()` to add this penalty.

<details>
<summary>Hint 1</summary>
Track `self._peak_value` in `reset()` and update it in `step()` after computing `portfolio_value`.
</details>

<details>
<summary>Hint 2 (more specific)</summary>
`drawdown = (self._peak_value - self._portfolio_value) / self._peak_value`. Add `-lambda * max(0, drawdown)` to the reward before returning.
</details>

In [ ]:
# YOUR CODE HERE
# ---------------

class TradingEnvWithDrawdown(TradingEnv):
    """
    TradingEnv with drawdown penalty in the reward.

    Parameters
    ----------
    drawdown_lambda : float
        Penalty coefficient for drawdown (higher = more risk-averse).
    """

    def __init__(self, prices, lookback=5, n_bins=5,
                 transaction_cost=0.001, drawdown_lambda=0.1):
        super().__init__(prices, lookback, n_bins, transaction_cost)
        self.drawdown_lambda = drawdown_lambda
        self._peak_value = 1.0

    def reset(self):
        """Reset including peak tracker."""
        result = super().reset()
        self._peak_value = 1.0
        return result

    def step(self, action: int) -> tuple:
        """
        Same as TradingEnv.step() but adds drawdown penalty to reward.
        """
        # YOUR CODE HERE: call super().step(), compute drawdown, modify reward
        pass


# Test the class
env_dd = TradingEnvWithDrawdown(TEST_PRICES)
state, _ = env_dd.reset()
next_s, r, term, trunc, info = env_dd.step(TradingEnv.LONG)
# Make sure it runs without error
print(f"Drawdown-penalized env: step reward = {r:.4f}")

In [ ]:
# AUTO-GRADED TESTS - Do not modify
# ----------------------------------
def test_exercise_9_4():
    # Verify the environment runs
    env_test = TradingEnvWithDrawdown(TEST_PRICES, drawdown_lambda=0.5)
    state, _ = env_test.reset()
    assert state is not None, "reset() must return a state"

    # Run several steps
    total_r = 0.0
    for _ in range(20):
        ns, r, term, trunc, info = env_test.step(TradingEnv.LONG)
        assert r is not None, "step() must return a reward"
        assert isinstance(r, (int, float)), f"Reward must be numeric, got {type(r)}"
        total_r += r
        if term or trunc:
            break

    # Verify drawdown penalty exists by comparing to standard env
    env_base = TradingEnv(TEST_PRICES)
    state_b, _ = env_base.reset()
    # Both envs start the same way — the difference appears when portfolio drops below peak
    # At step 1 (portfolio = initial), there's no drawdown yet
    env_dd_t = TradingEnvWithDrawdown(TEST_PRICES, drawdown_lambda=0.1)
    env_dd_t.reset()
    # Force portfolio below peak by simulating manually
    env_dd_t._portfolio_value = 0.9
    env_dd_t._peak_value = 1.0

    # Verify peak value is tracked
    assert hasattr(env_dd_t, '_peak_value'), "Must have _peak_value attribute"

    print("Exercise 9.4 passed! Drawdown-penalized trading environment correct.")

test_exercise_9_4()

In [ ]:
section_divider("Key Takeaways")

## Key Takeaways

1. **Trading is a natural RL problem** with well-defined state (market observables + position), actions (buy/hold/sell), and reward (portfolio returns). The challenge is non-stationarity and delayed reward.

2. **State discretization** enables tabular Q-learning on continuous price data. Momentum signals (recent returns) are the simplest predictive features; real systems use MACD, RSI, volatility estimates, and more.

3. **Transaction costs change optimal policy**: higher costs reduce optimal trading frequency. The agent learns to hold positions longer rather than rapidly switching.

4. **Reward shaping** allows encoding financial objectives beyond raw return: Sharpe ratio maximization, drawdown limits, and position concentration constraints can all be built into the reward function.

5. **Backtesting is necessary but not sufficient**: outperforming buy-and-hold on one test period may be luck. Real trading systems require walk-forward testing, transaction cost realism, and careful regime analysis.

## What's Next

This completes the core RL curriculum. The course capstone project applies the full pipeline — environment design, agent training, and evaluation — to a domain of your choice. The techniques from all nine modules (DP, MC, TD, function approximation, deep RL, policy gradients, advanced optimization, model-based RL, and frontiers) form the toolkit you now have.

## Additional Resources

- Mnih et al. (2015): DQN for Atari — the paper that started deep RL at scale
- Hambly, Xu, Yang (2023): "Recent Advances in Reinforcement Learning in Finance" — survey
- FinRL: https://github.com/AI4Finance-Foundation/FinRL — open-source RL trading framework
- Stable-Baselines3 + Gymnasium: industry-standard tools for production RL

In [ ]:
key_takeaways([
    "Review the key concepts covered in this notebook",
    "Practice applying these techniques to your own data",
    "Connect this material to the companion guide and slides"
])